# Combining Multi-Document Intelligence with User-Friendly Interface

This notebook demonstrates how to build a production-ready document Q&A system that combines:

- Intelligent multi-document detection and classification
- Query routing for improved retrieval accuracy
- Rich metadata preservation
- User-friendly Gradio interface

## Core Imports and Configuration

In [ ]:
import os
import asyncio
import nest_asyncio
import gradio as gr
from rank_bm25 import BM25Okapi
import fitz  # PyMuPDF
from PyPDF2 import PdfReader
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
import json
import re
from datetime import datetime
import hashlib

# Apply nest_asyncio for Jupyter/notebook async compatibility
try:
    loop = asyncio.get_event_loop()
except RuntimeError:
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
nest_asyncio.apply()

# -- Mistral via LlamaIndex (avoids direct mistralai import conflicts) --------
from llama_index.llms.mistralai import MistralAI

# -- LlamaIndex core imports --------------------------------------------------
from llama_index.core import Document, VectorStoreIndex, StorageContext, Settings
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator

In [ ]:
# -- API Key Configuration ----------------------------------------------------
# Loads MISTRAL_API_KEY from Colab secrets if available, otherwise falls back
# to a standard environment variable (useful for local / non-Colab runtimes).

def load_api_key(secret_name: str = "MISTRAL_API_KEY") -> str:
    """Load API key from Colab secrets or environment variables."""
    # 1. Try Google Colab secrets
    try:
        from google.colab import userdata
        key = userdata.get(secret_name)
        if key:
            print(f"Loaded {secret_name} from Colab secrets.")
            return key
    except Exception:
        pass  # Not running in Colab

    # 2. Fall back to environment variable
    key = os.environ.get(secret_name, "")
    if key:
        print(f"Loaded {secret_name} from environment variable.")
        return key

    print(f"  {secret_name} not found. Set it in Colab secrets or as an env var.")
    return ""

MISTRAL_API_KEY = load_api_key("MISTRAL_API_KEY")

In [ ]:
# -- Mistral Client & Model Wrapper  ------------------------------------------
MODEL_NAME = "mistral-small-latest"

llm = MistralAI(
    api_key=MISTRAL_API_KEY,
    model=MODEL_NAME,
)

# Quick connectivity check
_test = llm.complete("Reply with OK only.")
print(f"Mistral connected  |  model: {MODEL_NAME}  |  test: {_test.text.strip()}")


class MistralResponseWrapper:
    """Exposes a .text attribute so all downstream code works unchanged."""
    def __init__(self, text: str):
        self.text = text


class MistralModelWrapper:
    """Drop-in replacement - calls llm.complete() so retrieval logic is untouched."""
    def __init__(self, llm: MistralAI):
        self.llm = llm

    def generate_content(self, prompt: str) -> MistralResponseWrapper:
        response = self.llm.complete(prompt)
        return MistralResponseWrapper(response.text)


# Keep 'gemini_model' alias so nothing else in the notebook breaks
gemini_model = MistralModelWrapper(llm)

# -- Embedding Models ---------------------------------------------------------
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
llama_embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

Settings.llm = llm
Settings.embed_model = llama_embed_model

# Initialize docTR OCR predictor (loaded once to avoid per-page overhead)
try:
    from doctr.models import ocr_predictor
    doctr_predictor = ocr_predictor(pretrained=True)
    print("docTR OCR predictor loaded successfully.")
except Exception as e:
    doctr_predictor = None
    print(f"docTR failed to load, will fall back to pytesseract: {e}")

print("Embedding models loaded  |  all-MiniLM-L6-v2")

## Data Structures for Enhanced Document Management
Let's define our data structures to handle complex document metadata:

In [ ]:
@dataclass
class PageInfo:
    """Stores information about a single page"""
    page_num: int
    text: str
    doc_type: Optional[str] = None
    page_in_doc: int = 0

@dataclass
class LogicalDocument:
    """Represents a logical document within a PDF"""
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    chunks: List[Dict] = None

@dataclass
class ChunkMetadata:
    """Rich metadata for each chunk"""
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = None

## Document Intelligence Functions
These functions handle document classification and boundary detection:

In [ ]:
def classify_document_type(text: str, max_length: int = 1500) -> str:
    """
    Classify the document type based on its content.
    Uses LLM to intelligently identify document category.
    """
    # Truncate text if too long to avoid token limits
    text_sample = text[:max_length] if len(text) > max_length else text

    prompt = f"""You are a mortgage document analyst specializing in classifying pages extracted from large, multi-document mortgage loan packages. These packages are often scanned PDFs containing many different document types bundled together.

    Your task: classify the document excerpt below into EXACTLY ONE of these categories using the signals listed.

    CATEGORIES AND KEY SIGNALS:
    - Mortgage Contract: loan amount, interest rate, promissory note, deed of trust, borrower/lender obligations, monthly payment, maturity date, adjustable rate
    - Lender Fee Sheet: origination fee, appraisal fee, title insurance, closing costs, APR disclosure, loan estimate, itemized fees, settlement charges
    - Pay Slip: gross pay, net pay, year-to-date earnings, deductions, employer name, pay period, FICA, withholding
    - Bank Statement: account number, beginning/ending balance, deposits, withdrawals, transaction history, available balance, routing number
    - Tax Document: W-2, 1099, adjusted gross income, federal/state withholding, tax year, IRS, Schedule
    - Land Deed: grantor, grantee, legal description of property, parcel number, county recorder, conveyance, quitclaim
    - Contract: agreement between parties, terms and conditions, obligations, signatures, service agreement (non-mortgage)
    - Insurance: policy number, coverage amount, premium, policyholder, beneficiary, declarations page, homeowners/title insurance
    - Form: application fields, checkboxes, borrower information form, uniform residential loan application (1003), disclosure form
    - ID Document: driver's license, passport, date of birth, identification number, photo ID
    - Resume: work history, education, skills, employment dates, professional profile
    - Invoice: bill, amount due, vendor, payment terms, line items
    - Letter: salutation, body text, closing, correspondence, notification, approval/denial letter
    - Bank Statement: already listed above
    - Report: analysis, findings, summary, evaluation, credit report, appraisal report
    - Other: does not match any category above

    IMPORTANT RULES:
    - Base your classification ONLY on the text provided below - do not assume content not present
    - If multiple types seem possible, choose the one with the strongest signal words present
    - Mortgage-related documents should be classified as Mortgage Contract, Lender Fee Sheet, or Land Deed before defaulting to Contract or Other

    DOCUMENT EXCERPT:
    {text_sample}

    Respond with ONLY the category name from the list above. No explanation, no punctuation, no extra words."""

    try:
        response = gemini_model.generate_content(prompt)
        doc_type = response.text.strip()

        # Normalize the response
        valid_types = [
            'Resume', 'Contract', 'Mortgage Contract', 'Invoice', 'Pay Slip',
            'Lender Fee Sheet', 'Land Deed', 'Bank Statement', 'Tax Document',
            'Insurance', 'Report', 'Letter', 'Form', 'ID Document',
            'Medical', 'Other'
        ]

        # Find best match (case-insensitive)
        for valid_type in valid_types:
            if doc_type.lower() == valid_type.lower():
                return valid_type

        return 'Other'
    except Exception as e:
        print(f"Classification error: {e}")
        return 'Other'

def detect_document_boundary(prev_text: str, curr_text: str,
                            current_doc_type: str = None) -> bool:
    """
    Detect if two consecutive pages belong to the same document.
    Returns True if they're from the same document.
    """
    # Quick heuristic checks first
    if not prev_text or not curr_text:
        return False

    # Sample the texts for LLM analysis
    prev_sample = prev_text[-500:] if len(prev_text) > 500 else prev_text
    curr_sample = curr_text[:500] if len(curr_text) > 500 else curr_text

    prompt = f"""You are analyzing pages extracted from a large mortgage loan package - a multi-document PDF "blob" where many different mortgage documents have been scanned together into one file.

    Your task: determine whether the page ending below and the page starting below belong to the SAME document or are the START of a NEW document.

    CURRENT DOCUMENT TYPE: {current_doc_type or 'Unknown'}

    --- END OF PREVIOUS PAGE ---
    {prev_sample}

    --- START OF NEXT PAGE ---
    {curr_sample}

    SIGNALS THAT INDICATE A NEW DOCUMENT IS STARTING:
    - A new document title or header appears (e.g. "Promissory Note", "Loan Estimate", "W-2 Tax Form")
    - Page numbering resets (e.g. "Page 1 of 3" after previously seeing "Page 3 of 3")
    - A different borrower name, loan number, or property address appears
    - A notary block, signature block, or "IN WITNESS WHEREOF" ends the previous page
    - The document type clearly shifts (e.g. financial table to legal agreement text)
    - A new form number or regulatory disclosure header appears

    SIGNALS THAT INDICATE THE SAME DOCUMENT CONTINUES:
    - Content, topic, or numbered sections continue logically
    - Same loan number, borrower name, or property address
    - Page numbering continues sequentially
    - Sentence or paragraph is clearly mid-thought from previous page
    - Same formatting, font style, or table structure

    Answer with ONLY one word - Yes if same document, No if new document starts."""

    try:
        response = gemini_model.generate_content(prompt)
        return response.text.strip().lower().startswith('yes')
    except Exception as e:
        print(f"Boundary detection error: {e}")
        # Default to keeping pages together if uncertain
        return True

## Advanced PDF Processing Pipeline
Now let's build the enhanced PDF processing pipeline:

In [ ]:
def extract_and_analyze_pdf(pdf_file) -> Tuple[List[PageInfo], List[LogicalDocument]]:
    """
    Extract text from PDF and perform intelligent document analysis.
    Returns both page-level info and logical document groupings.
    Supports various file types including scanned PDFs with OCR.
    """
    print("Starting PDF extraction and analysis...")

    # Extract text from each page
    if isinstance(pdf_file, dict) and "content" in pdf_file:
        doc = fitz.open(stream=pdf_file["content"], filetype="pdf")
    elif hasattr(pdf_file, "read"):
        doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
    else:
        doc = fitz.open(pdf_file)

    pages_info = []
    for i, page in enumerate(doc):
        text = page.get_text()

        # If no text found, try OCR (for scanned documents)
        if not text.strip():
            print(f"  Page {i}: No text found, attempting OCR...")
            ocr_success = False

            # --- Primary OCR: docTR ---
            if doctr_predictor is not None:
                try:
                    import tempfile, os
                    from PIL import Image
                    import io

                    pix = page.get_pixmap(dpi=200)  # Higher DPI = better OCR accuracy
                    img_data = pix.tobytes("png")

                    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
                        tmp_path = tmp.name
                        tmp.write(img_data)

                    from doctr.io import DocumentFile
                    doctr_doc = DocumentFile.from_images([tmp_path])
                    result = doctr_predictor(doctr_doc)

                    # Flatten all recognized words in reading order
                    words = []
                    for page_result in result.pages:
                        for block in page_result.blocks:
                            for line in block.lines:
                                words.extend([w.value for w in line.words])

                    text = " ".join(words)
                    os.unlink(tmp_path)

                    if text.strip():
                        print(f"  Page {i}: docTR OCR succeeded ({len(text)} chars)")
                        ocr_success = True
                    else:
                        print(f"  Page {i}: docTR returned empty text, trying pytesseract fallback...")

                except Exception as e:
                    print(f"  Page {i}: docTR OCR failed - {e}. Trying pytesseract fallback...")

            # --- Fallback OCR: pytesseract ---
            if not ocr_success:
                try:
                    from PIL import Image
                    import pytesseract
                    import io

                    pix = page.get_pixmap()
                    img_data = pix.tobytes("png")
                    img = Image.open(io.BytesIO(img_data))
                    text = pytesseract.image_to_string(img)

                    if text.strip():
                        print(f"  Page {i}: pytesseract OCR succeeded ({len(text)} chars)")
                    else:
                        print(f"  Page {i}: Both OCR methods returned empty text.")

                except Exception as e:
                    print(f"  Page {i}: pytesseract OCR also failed - {e}")
                    text = ""

        pages_info.append(PageInfo(page_num=i, text=text))

    doc.close()

    if not pages_info:
        raise ValueError("No text could be extracted from PDF")

    print(f"Extracted {len(pages_info)} pages")

    # Perform document classification and boundary detection
    print("Analyzing document structure...")
    logical_docs = []
    current_doc_type = None
    current_doc_pages = []
    doc_counter = 0

    for i, page_info in enumerate(pages_info):
        if i == 0:
            # First page - classify document type
            current_doc_type = classify_document_type(page_info.text)
            page_info.doc_type = current_doc_type
            page_info.page_in_doc = 0
            current_doc_pages = [page_info]
            print(f"  Page {i}: New document detected - {current_doc_type}")
        else:
            # Check if this page continues the previous document
            prev_text = pages_info[i-1].text
            is_same = detect_document_boundary(prev_text, page_info.text, current_doc_type)

            if is_same:
                # Continue current document
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = len(current_doc_pages)
                current_doc_pages.append(page_info)
            else:
                # New document detected - save previous and start new
                logical_doc = LogicalDocument(
                    doc_id=f"doc_{doc_counter}",
                    doc_type=current_doc_type,
                    page_start=current_doc_pages[0].page_num,
                    page_end=current_doc_pages[-1].page_num,
                    text="\n\n".join([p.text for p in current_doc_pages])
                )
                logical_docs.append(logical_doc)
                doc_counter += 1

                # Start new document
                current_doc_type = classify_document_type(page_info.text)
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = 0
                current_doc_pages = [page_info]
                print(f"  Page {i}: New document detected - {current_doc_type}")

    # Don't forget the last document
    if current_doc_pages:
        logical_doc = LogicalDocument(
            doc_id=f"doc_{doc_counter}",
            doc_type=current_doc_type,
            page_start=current_doc_pages[0].page_num,
            page_end=current_doc_pages[-1].page_num,
            text="\n\n".join([p.text for p in current_doc_pages])
        )
        logical_docs.append(logical_doc)

    print(f"Identified {len(logical_docs)} logical documents")
    for ld in logical_docs:
        print(f"   - {ld.doc_type}: Pages {ld.page_start}-{ld.page_end}")

    return pages_info, logical_docs

## Intelligent Chunking with Metadata Preservation
We'll provide two chunking approaches - our custom implementation and LlamaIndex's built-in capabilities:

In [ ]:
def chunk_document_with_metadata(logical_doc: LogicalDocument,
                                chunk_size: int = 500,
                                overlap: int = 100) -> List[ChunkMetadata]:
    """
    Chunk a logical document while preserving rich metadata.
    Uses sliding window with overlap for better context.
    """
    chunks_metadata = []
    words = logical_doc.text.split()

    if len(words) <= chunk_size:
        # Document is small enough to be a single chunk
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_0",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=0,
            page_start=logical_doc.page_start,
            page_end=logical_doc.page_end,
            text=logical_doc.text
        )
        chunks_metadata.append(chunk_meta)
    else:
        # Create overlapping chunks
        stride = chunk_size - overlap
        for i, start_idx in enumerate(range(0, len(words), stride)):
            end_idx = min(start_idx + chunk_size, len(words))
            chunk_text = ' '.join(words[start_idx:end_idx])

            # Calculate which pages this chunk spans
            # (simplified - in production, track more precisely)
            chunk_position = start_idx / len(words)
            page_range = logical_doc.page_end - logical_doc.page_start
            relative_page = int(chunk_position * page_range)
            chunk_page_start = logical_doc.page_start + relative_page
            chunk_page_end = min(chunk_page_start + 1, logical_doc.page_end)

            chunk_meta = ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=i,
                page_start=chunk_page_start,
                page_end=chunk_page_end,
                text=chunk_text
            )
            chunks_metadata.append(chunk_meta)

            if end_idx >= len(words):
                break

    return chunks_metadata

def chunk_with_llama_index(logical_doc: LogicalDocument,
                           chunk_size: int = 500,
                           chunk_overlap: int = 100) -> List[Document]:
    """
    Alternative: Use LlamaIndex's advanced chunking with metadata.
    """
    # Create LlamaIndex document with metadata
    doc = Document(
        text=logical_doc.text,
        metadata={
            "doc_id": logical_doc.doc_id,
            "doc_type": logical_doc.doc_type,
            "page_start": logical_doc.page_start,
            "page_end": logical_doc.page_end,
            "source": f"{logical_doc.doc_type}_document"
        }
    )

    # Use LlamaIndex's sentence splitter for better chunking
    splitter = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        paragraph_separator="\n\n",
        separator=" ",
    )

    # Create nodes (chunks) from document
    nodes = splitter.get_nodes_from_documents([doc])

    # Convert to our ChunkMetadata format for consistency
    chunks_metadata = []
    for i, node in enumerate(nodes):
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=i,
            page_start=node.metadata.get("page_start", logical_doc.page_start),
            page_end=node.metadata.get("page_end", logical_doc.page_end),
            text=node.text
        )
        chunks_metadata.append(chunk_meta)

    return chunks_metadata

def process_all_documents(logical_docs: List[LogicalDocument],
                         use_llama_index: bool = False) -> List[ChunkMetadata]:
    """
    Process all logical documents into chunks with metadata.
    Can use either custom or LlamaIndex chunking.
    """
    all_chunks = []

    for logical_doc in logical_docs:
        if use_llama_index:
            chunks = chunk_with_llama_index(logical_doc)
        else:
            chunks = chunk_document_with_metadata(logical_doc)

        logical_doc.chunks = chunks  # Store reference
        all_chunks.extend(chunks)
        print(f"{logical_doc.doc_type}: Created {len(chunks)} chunks")

    return all_chunks

## Query Routing and Intelligent Retrieval

In [ ]:
# -- JSON extraction helper  --------------------------------------------------
def _extract_json(text: str) -> str:
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fenced:
        return fenced.group(1)
    bare = re.search(r"\{.*\}", text, re.DOTALL)
    if bare:
        return bare.group(0)
    return text.strip()


# -- Query routing  -----------------------------------------------------------
def predict_query_document_type(query: str) -> Tuple[str, float]:
    prompt = f"""You are a mortgage document routing specialist. A user is asking a question about their mortgage loan package. Your job is to predict which document type most likely contains the answer, so the system can search that document first.

USER QUERY: "{query}"

DOCUMENT TYPES AND THE QUESTIONS THEY TYPICALLY ANSWER:
- Mortgage Contract: interest rate, loan term, monthly payment amount, prepayment penalty, late fees, balloon payment, ARM adjustment caps, what happens if I miss a payment
- Lender Fee Sheet: closing costs, origination fees, appraisal fee, title insurance cost, APR, total loan costs, itemized fees at closing, points
- Pay Slip: gross income, net pay, employer, deductions, pay frequency, year-to-date earnings, overtime
- Bank Statement: account balance, recent transactions, deposits, withdrawals, available funds, direct deposits
- Tax Document: annual income, tax withheld, W-2 wages, 1099 income, adjusted gross income, tax year totals
- Land Deed: property address, legal description, ownership history, parcel number, property boundaries, grantor/grantee
- Insurance: coverage amount, policy number, premium, what is covered, homeowners insurance, flood insurance
- Form: borrower information, application data, uniform residential loan application fields, disclosures signed
- Contract: service agreements, non-mortgage legal obligations, vendor terms
- Letter: approval or denial status, communication from lender, rate lock confirmation, conditions of approval
- Report: credit score, appraisal value, property condition, comparable sales, inspection findings
- Other: general questions not clearly tied to a specific document type

RULES:
- Assign high confidence (0.8-1.0) only when the query contains clear mortgage-specific terminology matching a document type
- Assign medium confidence (0.5-0.79) when the query is related but ambiguous
- Assign low confidence (0.0-0.49) when the query is vague or could match multiple types
- Do not guess - low confidence with a broad search is better than a wrong high-confidence routing

Respond with ONLY a valid JSON object, no explanation, no markdown:
{{\"type\": \"DocumentType\", \"confidence\": 0.85}}"""

    try:
        response = gemini_model.generate_content(prompt)
        raw = response.text.strip()
        cleaned = _extract_json(raw)
        result = json.loads(cleaned)
        return result.get("type", "Other"), float(result.get("confidence", 0.5))
    except Exception as e:
        print(f"Query routing error: {e}")
        return "Other", 0.0


# -- Score normalization  -----------------------------------------------------
def min_max_normalize_scores(scores: Dict[str, float]) -> Dict[str, float]:
    """Normalize a dict of scores to [0, 1] range."""
    if not scores:
        return {}
    min_score = min(scores.values())
    max_score = max(scores.values())
    if max_score == min_score:
        return {k: 0.0 for k in scores}
    return {k: (v - min_score) / (max_score - min_score) for k, v in scores.items()}


# -- Query expansion  ---------------------------------------------------------
def generate_query_variants(query: str, num_variants: int = 3) -> List[str]:
    """Use the LLM to generate alternative phrasings of the query.
    More query variants = broader retrieval coverage = more robust answers."""
    prompt = f"""You are a mortgage document search assistant helping retrieve information from mortgage loan packages.

Generate {num_variants - 1} alternative phrasings of the following question. Use different mortgage terminology where appropriate to maximize the chance of finding relevant document sections.

Original question: {query}

Rules:
- Return ONLY the alternative phrasings, one per line
- No numbering, no labels, no explanation
- Keep the same intent - these are rephrasings, not new questions"""

    try:
        response = gemini_model.generate_content(prompt)
        variants = [v.strip() for v in response.text.strip().split('\n') if v.strip()]
        return [query] + variants[:num_variants - 1]
    except Exception as e:
        print(f"Query expansion error: {e}")
        return [query]


# -- Reciprocal Rank Fusion  --------------------------------------------------
def reciprocal_rank_fusion(
    ranked_lists: List[List[Tuple]], k: int = 60
) -> List[Tuple]:
    """Combine multiple ranked retrieval lists using Reciprocal Rank Fusion.
    k=60 is the standard RRF constant that prevents top-ranked results from
    dominating when they appear in only one list."""
    rrf_scores: Dict[str, float] = {}
    chunk_map: Dict[str, object] = {}

    for ranked_list in ranked_lists:
        for rank, (chunk_meta, _score) in enumerate(ranked_list):
            cid = chunk_meta.chunk_id
            if cid not in rrf_scores:
                rrf_scores[cid] = 0.0
                chunk_map[cid] = chunk_meta
            rrf_scores[cid] += 1.0 / (k + rank + 1)

    sorted_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    return [(chunk_map[cid], rrf_scores[cid]) for cid in sorted_ids]


# -- Retriever: hybrid FAISS + BM25, query expansion, RRF --------------------
class IntelligentRetriever:
    """
    Retrieval system combining:
    - Semantic search (FAISS vector index)
    - Keyword search (BM25)
    - Query expansion (LLM-generated query variants)
    - Reciprocal Rank Fusion (merges results across all variants)
    - Per-doc-type sub-indices for metadata-filtered retrieval
    """

    def __init__(self):
        self.index = None
        self.chunks_metadata = []
        self.doc_type_indices = {}
        self.bm25_index = None
        self.bm25_doc_type_indices = {}
        self.cache_hits = 0
        self.total_queries = 0

    def build_indices(self, chunks_metadata: List[ChunkMetadata]):
        print("Building vector + BM25 indices...")
        self.chunks_metadata = chunks_metadata

        texts = [chunk.text for chunk in chunks_metadata]

        # -- FAISS vector index -----------------------------------------------
        embeddings = embed_model.encode(texts, show_progress_bar=True)
        embeddings = np.array(embeddings, dtype="float32")
        for i, chunk in enumerate(chunks_metadata):
            chunk.embedding = embeddings[i]

        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)

        # -- BM25 keyword index -----------------------------------------------
        tokenized_corpus = [text.lower().split() for text in texts]
        self.bm25_index = BM25Okapi(tokenized_corpus)

        # -- Per-doc-type sub-indices ------------------------------------------
        doc_types = set(chunk.doc_type for chunk in chunks_metadata)
        for doc_type in doc_types:
            type_indices = [
                i for i, chunk in enumerate(chunks_metadata)
                if chunk.doc_type == doc_type
            ]
            if type_indices:
                # FAISS sub-index
                type_embeddings = embeddings[type_indices]
                type_index = faiss.IndexFlatL2(dim)
                type_index.add(type_embeddings)
                self.doc_type_indices[doc_type] = {
                    'index': type_index,
                    'mapping': type_indices
                }
                # BM25 sub-index
                type_corpus = [tokenized_corpus[i] for i in type_indices]
                self.bm25_doc_type_indices[doc_type] = {
                    'bm25': BM25Okapi(type_corpus),
                    'mapping': type_indices
                }

        print(f"Indexed {len(chunks_metadata)} chunks across {len(doc_types)} "
              f"document types (vector + BM25)")

    def _hybrid_retrieve_single(
        self, query: str, k: int,
        filter_doc_type: Optional[str] = None,
        alpha: float = 0.6
    ) -> List[Tuple[ChunkMetadata, float]]:
        """Hybrid retrieval for one query: FAISS (semantic) + BM25 (keyword),
        combined with weighted alpha and min-max normalisation.
        alpha=0.6 weights semantic search slightly higher than keyword search."""

        query_embedding = np.array(embed_model.encode([query]), dtype="float32")
        tokenized_query = query.lower().split()

        # -- FAISS retrieval --------------------------------------------------
        if filter_doc_type and filter_doc_type in self.doc_type_indices:
            type_data = self.doc_type_indices[filter_doc_type]
            k_capped = min(k, type_data['index'].ntotal)
            D, I = type_data['index'].search(query_embedding, k_capped)
            faiss_chunk_indices = [type_data['mapping'][i] for i in I[0]]
            faiss_distances = D[0]
        else:
            k_capped = min(k, self.index.ntotal)
            D, I = self.index.search(query_embedding, k_capped)
            faiss_chunk_indices = [i for i in I[0] if i < len(self.chunks_metadata)]
            faiss_distances = D[0][:len(faiss_chunk_indices)]

        max_dist = max(faiss_distances) if len(faiss_distances) > 0 and max(faiss_distances) > 0 else 1.0
        vector_scores = {
            self.chunks_metadata[ci].chunk_id: (max_dist - d) / max_dist
            for ci, d in zip(faiss_chunk_indices, faiss_distances)
            if ci < len(self.chunks_metadata)
        }

        # -- BM25 retrieval ---------------------------------------------------
        keyword_scores: Dict[str, float] = {}
        if filter_doc_type and filter_doc_type in self.bm25_doc_type_indices:
            bm25_data = self.bm25_doc_type_indices[filter_doc_type]
            bm25_raw = bm25_data['bm25'].get_scores(tokenized_query)
            for local_i, global_i in enumerate(bm25_data['mapping']):
                if global_i < len(self.chunks_metadata):
                    cid = self.chunks_metadata[global_i].chunk_id
                    keyword_scores[cid] = float(bm25_raw[local_i])
        else:
            bm25_raw = self.bm25_index.get_scores(tokenized_query)
            for global_i, score in enumerate(bm25_raw):
                if global_i < len(self.chunks_metadata):
                    cid = self.chunks_metadata[global_i].chunk_id
                    keyword_scores[cid] = float(score)

        # -- Normalise and combine --------------------------------------------
        norm_vector = min_max_normalize_scores(vector_scores)
        norm_keyword = min_max_normalize_scores(keyword_scores)

        chunk_id_map = {chunk.chunk_id: chunk for chunk in self.chunks_metadata}
        all_chunk_ids = set(norm_vector) | set(norm_keyword)

        results = []
        for cid in all_chunk_ids:
            if cid in chunk_id_map:
                combined = (alpha * norm_vector.get(cid, 0.0)
                            + (1 - alpha) * norm_keyword.get(cid, 0.0))
                results.append((chunk_id_map[cid], combined))

        results.sort(key=lambda x: x[1], reverse=True)
        return results[:k]

    def retrieve(
        self, query: str, k: int = 4,
        filter_doc_type: Optional[str] = None,
        auto_route: bool = True,
        use_query_expansion: bool = True
    ) -> List[Tuple[ChunkMetadata, float]]:
        """
        Full retrieval pipeline:
        1. Auto-route query to the most likely document type
        2. Expand query into multiple variants using the LLM
        3. Run hybrid retrieval for each variant
        4. Fuse all ranked lists with Reciprocal Rank Fusion
        """
        self.total_queries += 1

        # -- Step 1: Query routing --------------------------------------------
        routed_type = filter_doc_type
        if routed_type is None and auto_route:
            predicted_type, confidence = predict_query_document_type(query)
            print(f"Routed to: {predicted_type} (confidence: {confidence:.2f})")
            if confidence > 0.7 and predicted_type in self.doc_type_indices:
                routed_type = predicted_type

        # -- Step 2: Query expansion ------------------------------------------
        if use_query_expansion:
            query_variants = generate_query_variants(query, num_variants=3)
            print(f"Expanded to {len(query_variants)} query variants")
        else:
            query_variants = [query]

        # -- Step 3: Hybrid retrieval per variant -----------------------------
        all_ranked_lists = [
            self._hybrid_retrieve_single(variant, k, routed_type)
            for variant in query_variants
        ]

        # -- Step 4: Reciprocal Rank Fusion -----------------------------------
        if len(all_ranked_lists) > 1:
            fused = reciprocal_rank_fusion(all_ranked_lists)
            print(f"RRF fused {len(all_ranked_lists)} ranked lists -> {len(fused)} unique chunks")
        else:
            fused = all_ranked_lists[0]

        return fused[:k]

## Enhanced Answer Generation with Source Attribution

In [ ]:
def generate_answer_with_sources(query: str,
                                retrieved_chunks: List[Tuple[ChunkMetadata, float]]) -> Dict:
    """
    Generate answer with detailed source attribution.
    """
    if not retrieved_chunks:
        return {
            'answer': "I couldn't find relevant information to answer your question.",
            'sources': [],
            'confidence': 0.0
        }

    # Prepare context from retrieved chunks
    context_parts = []
    sources = []

    for chunk_meta, score in retrieved_chunks:
        context_parts.append(f"[From {chunk_meta.doc_type}, Pages {chunk_meta.page_start}-{chunk_meta.page_end}]")
        context_parts.append(chunk_meta.text)
        context_parts.append("")

        sources.append({
            'doc_type': chunk_meta.doc_type,
            'pages': f"{chunk_meta.page_start}-{chunk_meta.page_end}",
            'relevance': f"{score:.2%}",
            'preview': chunk_meta.text[:100] + "..."
        })

    context = "\n".join(context_parts)

    # Generate answer
    prompt = f"""You are a mortgage document intelligence assistant built to help users extract accurate information from their mortgage loan package. This package may contain many document types: contracts, fee sheets, pay slips, bank statements, tax documents, deeds, and more.

    RETRIEVED DOCUMENT CONTEXT:
    {context}

    USER QUESTION: {query}

    YOUR INSTRUCTIONS:
    1. Answer using ONLY the information present in the retrieved context above - do not use outside knowledge
    2. If the exact answer is present, state it clearly and cite the document type and page number (e.g. "According to the Mortgage Contract on pages 3-4...")
    3. If the context contains partial information, share what is available and clearly state what is missing
    4. If the context does not contain enough information to answer, say exactly: "The uploaded documents do not appear to contain a clear answer to this question. You may want to check a different section of your loan package or consult your lender directly."
    5. Never invent numbers, dates, names, loan amounts, interest rates, or legal terms - accuracy is critical in mortgage documents
    6. Use plain language - assume the user is a homeowner, not a mortgage professional
    7. If the answer involves a dollar amount, interest rate, or legal obligation, quote it exactly as it appears in the document rather than paraphrasing

    Answer:"""

    try:
        response = gemini_model.generate_content(prompt)
        answer = response.text.strip()

        # Calculate overall confidence based on retrieval scores
        avg_score = sum(s for _, s in retrieved_chunks) / len(retrieved_chunks)

        return {
            'answer': answer,
            'sources': sources,
            'confidence': avg_score,
            'chunks_used': len(retrieved_chunks)
        }
    except Exception as e:
        print(f"Answer generation error: {e}")
        return {
            'answer': f"Error generating answer: {str(e)}",
            'sources': sources,
            'confidence': 0.0
        }

## Enhanced Document Store

In [ ]:
class EnhancedDocumentStore:
    """
    Manages the complete document processing and retrieval pipeline.
    Processes a single PDF (which may contain multiple logical documents)
    and exposes a unified query interface over the full contents.
    """

    def __init__(self):
        self.pages_info = []
        self.logical_docs = []
        self.chunks_metadata = []
        self.retriever = IntelligentRetriever()
        self.is_ready = False
        self.processing_stats = {}
        self.filename = None

    def process_pdf(self, pdf_file, filename: str = "document.pdf"):
        """
        Complete PDF processing pipeline.
        Uses LlamaIndex SentenceSplitter chunking for better structure-awareness.
        """
        self.filename = filename
        self.is_ready = False
        start_time = datetime.now()

        try:
            # Step 1: Extract text + identify logical document boundaries
            self.pages_info, self.logical_docs = extract_and_analyze_pdf(pdf_file)

            # Step 2: Chunk with LlamaIndex SentenceSplitter (use_llama_index=True)
            # LlamaIndex respects sentence boundaries and paragraph breaks, which
            # is important for structured legal/mortgage documents where splitting
            # mid-clause degrades retrieval quality.
            self.chunks_metadata = process_all_documents(self.logical_docs, use_llama_index=True)

            # Step 3: Build hybrid FAISS + BM25 retrieval indices
            self.retriever.build_indices(self.chunks_metadata)

            process_time = (datetime.now() - start_time).total_seconds()
            self.processing_stats = {
                'filename': filename,
                'total_pages': len(self.pages_info),
                'documents_found': len(self.logical_docs),
                'total_chunks': len(self.chunks_metadata),
                'document_types': list(set(doc.doc_type for doc in self.logical_docs)),
                'processing_time': f"{process_time:.1f}s"
            }

            self.is_ready = True
            return True, self.processing_stats

        except Exception as e:
            return False, {'error': str(e)}

    def query(self, question: str, filter_type=None,
              auto_route: bool = True, k: int = 4):
        """Query the document store."""
        if not self.is_ready:
            return {
                'answer': "Please upload and process a PDF first.",
                'sources': [],
                'confidence': 0.0
            }

        retrieved = self.retriever.retrieve(
            question, k=k,
            filter_doc_type=filter_type,
            auto_route=auto_route
        )

        result = generate_answer_with_sources(question, retrieved)
        result['filter_used'] = filter_type or ('auto' if auto_route else 'none')
        return result

    def get_document_structure(self):
        """Get the document structure for UI display."""
        if not self.logical_docs:
            return []

        structure = []
        for doc in self.logical_docs:
            structure.append({
                'id': doc.doc_id,
                'type': doc.doc_type,
                'pages': f"{doc.page_start + 1}-{doc.page_end + 1}",
                'chunks': len(doc.chunks) if doc.chunks else 0,
                'preview': doc.text[:200] + "..." if len(doc.text) > 200 else doc.text
            })
        return structure


## Gradio Interface with Enhanced Features
Now let's create the sophisticated Gradio interface:

In [ ]:
# -- Global document store (single-PDF) ---------------------------------------
doc_store = EnhancedDocumentStore()


# -- PDF processing handler ---------------------------------------------------
def process_pdf_handler(pdf_file):
    """Handle single PDF upload and kick off the full processing pipeline."""
    if pdf_file is None:
        return "Please upload a PDF file.", "", gr.update(choices=["All"])

    # Gradio 6.x passes filepath as a plain string for type="filepath"
    filepath = pdf_file if isinstance(pdf_file, str) else getattr(pdf_file, "name", str(pdf_file))
    filename = os.path.basename(filepath)

    success, stats = doc_store.process_pdf(filepath, filename=filename)

    if success:
        status_msg = f"""
Successfully Processed:
- File(s): {stats['filename']}
- Pages: {stats['total_pages']}
- Documents Found: {stats['documents_found']}
- Chunks Created: {stats['total_chunks']}
- Types: {', '.join(stats['document_types'])}
- Time: {stats['processing_time']}
        """.strip()

        structure = doc_store.get_document_structure()
        structure_display = "\n".join([
            f"- **{doc['type']}** (Pages {doc['pages']}): {doc['chunks']} chunks"
            for doc in structure
        ])
        doc_types = ["All"] + stats["document_types"]
        return status_msg, structure_display, gr.update(choices=doc_types, value="All")
    else:
        return f"Error: {stats.get('error', 'Unknown error')}", "", gr.update(choices=["All"])


# -- Chat handler -------------------------------------------------------------
def chat_handler(message, history, doc_filter, auto_route, num_chunks):
    """
    Handle a single user message. Appends user + assistant turns to history
    and returns the updated list. Works for any number of prior turns.
    """
    history = history or []

    if not message or not message.strip():
        return history

    try:
        if not doc_store.is_ready:
            response = "Please upload and process a PDF document first."
        else:
            filter_type = None if doc_filter == "All" else doc_filter
            result = doc_store.query(
                message,
                filter_type=filter_type,
                auto_route=auto_route and (filter_type is None),
                k=num_chunks
            )

            response = result["answer"]

            if result.get("sources"):
                response += "\n\n**Sources:**\n"
                for src in result["sources"]:
                    response += f"- {src['doc_type']} (Pages {src['pages']}) - Relevance: {src['relevance']}\n"

            response += f"\n*Confidence: {result['confidence']:.1%} | Filter: {result.get('filter_used', 'none')}*"

    except Exception as exc:
        response = f"Something went wrong while processing your query: {exc}"

    history.append({"role": "user",      "content": message})
    history.append({"role": "assistant", "content": response})
    return history


# -- Gradio interface ---------------------------------------------------------
def create_interface():
    with gr.Blocks(title="Enhanced Document Q&A", theme=gr.themes.Glass()) as demo:

        gr.Markdown("""
        # Enhanced Document Q&A System
        ### Intelligent Multi-Document Analysis with Advanced RAG Pipeline
        """)

        with gr.Row():

            # -- Left column: upload + controls --------------------------------
            with gr.Column(scale=2):
                pdf_input = gr.File(
                    label="Upload PDF Document",
                    file_types=[".pdf"],
                    type="filepath",
                    height=120
                )
                with gr.Row():
                    process_btn   = gr.Button("Process Document", variant="primary",   size="lg", scale=2)
                    clear_all_btn = gr.Button("Clear All",        variant="secondary", size="lg", scale=1)

            # -- Middle column: info + settings --------------------------------
            with gr.Column(scale=1):
                gr.Markdown("### Document Info")
                status_output    = gr.Markdown(value="Waiting for PDF upload...")
                structure_output = gr.Markdown(value="")

                gr.Markdown("### Settings")
                doc_filter = gr.Dropdown(
                    choices=["All"], value="All",
                    label="Document Type Filter",
                    info="Filter search to a specific document type"
                )
                auto_route = gr.Checkbox(
                    value=True,
                    label="Auto-Route Queries",
                    info="Automatically detect the most relevant document type"
                )
                num_chunks = gr.Slider(minimum=1, maximum=10, value=4, step=1,
                                       label="Chunks to Retrieve")

            # -- Right column: chat -------------------------------------------
            with gr.Column(scale=2):
                gr.Markdown("### Ask Questions")
                chatbot = gr.Chatbot(label="Conversation", height=500, show_label=False, type="messages")
                with gr.Row():
                    msg_input = gr.Textbox(
                        placeholder="Ask anything about your mortgage documents...",
                        scale=4, show_label=False
                    )
                    send_btn = gr.Button("Send", scale=1, variant="primary")
                with gr.Row():
                    clear_chat_btn = gr.Button("Clear Chat",          size="sm", scale=1)
                    example_btn1   = gr.Button("What's the summary?", size="sm", scale=1)
                    example_btn2   = gr.Button("Find amounts",        size="sm", scale=1)

        # -- Status bar -------------------------------------------------------
        with gr.Row():
            status_bar = gr.Markdown(value="**Status:** Ready | **Documents:** 0 | **Chunks:** 0")

        # -- Internal helpers -------------------------------------------------
        def update_status_bar():
            if doc_store.is_ready:
                stats    = doc_store.processing_stats
                total_q  = doc_store.retriever.total_queries
                hits     = doc_store.retriever.cache_hits
                cache_rt = (hits / total_q * 100) if total_q > 0 else 0
                return (f"**Status:** Ready | "
                        f"**Documents:** {stats.get('documents_found', 0)} | "
                        f"**Chunks:** {stats.get('total_chunks', 0)} | "
                        f"**Cache Rate:** {cache_rt:.0f}%")
            return "**Status:** Ready | **Documents:** 0 | **Chunks:** 0"

        def clear_all():
            global doc_store
            doc_store = EnhancedDocumentStore()
            return (None, "Waiting for PDF upload...", "",
                    gr.update(choices=["All"], value="All"), [], "", update_status_bar())

        def process_pdf_with_status(pdf_file):
            status, structure, filter_update = process_pdf_handler(pdf_file)
            return status, structure, filter_update, update_status_bar()

        def chat_with_status(message, history, df, ar, nc):
            """Always returns a 2-tuple (history, status_text) regardless of errors."""
            try:
                new_history = chat_handler(message, history, df, ar, nc)
            except Exception as exc:
                new_history = list(history or [])
                new_history.append({"role": "user",      "content": message or ""})
                new_history.append({"role": "assistant", "content": f"Unexpected error: {exc}"})
            return new_history, update_status_bar()

        def ask_summary(history):
            new_history, status = chat_with_status(
                "Can you provide a summary of the main points in this document?",
                history, "All", True, 4
            )
            return new_history, status

        def ask_amounts(history):
            new_history, status = chat_with_status(
                "What are all the monetary amounts or financial figures mentioned?",
                history, "All", True, 4
            )
            return new_history, status

        # -- Event wiring -----------------------------------------------------
        process_btn.click(
            fn=process_pdf_with_status, inputs=[pdf_input],
            outputs=[status_output, structure_output, doc_filter, status_bar]
        )

        clear_all_btn.click(
            fn=clear_all,
            outputs=[pdf_input, status_output, structure_output,
                     doc_filter, chatbot, msg_input, status_bar]
        )

        # Both Submit (Enter key) and Send button wire the same function
        for trigger in [msg_input.submit, send_btn.click]:
            trigger(
                fn=chat_with_status,
                inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
                outputs=[chatbot, status_bar]
            ).then(lambda: "", outputs=[msg_input])

        clear_chat_btn.click(lambda: [], outputs=[chatbot])

        example_btn1.click(fn=ask_summary, inputs=[chatbot], outputs=[chatbot, status_bar])
        example_btn2.click(fn=ask_amounts, inputs=[chatbot], outputs=[chatbot, status_bar])

    return demo


In [ ]:
demo = create_interface()
demo.queue()
demo.launch()